In [ ]:
# ========== CELL 1: IMPORT + CONNECT ==========
import odrive
import time

print("🔌 Connecting to ODrive...")
try:
    odrv0 = odrive.find_any()
    print(f"✅ Connected! VBUS: {odrv0.vbus_voltage:.1f}V")
except:
    print("❌ No ODrive found - check USB!")
    raise


🔌 Connecting to ODrive...
USB device init failed (bus 3, device 108). Ignoring this device. More info: Traceback (most recent call last):
  File "/home/aida/micromamba/envs/odroid/lib/python3.11/site-packages/fibre/usbbulk_transport.py", line 196, in discover_channels
    bulk_device.init()
  File "/home/aida/micromamba/envs/odroid/lib/python3.11/site-packages/fibre/usbbulk_transport.py", line 50, in init
    self.dev.reset()
  File "/home/aida/micromamba/envs/odroid/lib/python3.11/site-packages/usb/core.py", line 975, in reset
    self._ctx.backend.reset_device(self._ctx.handle)
  File "/home/aida/micromamba/envs/odroid/lib/python3.11/site-packages/usb/backend/libusb1.py", line 915, in reset_device
    _check(self.lib.libusb_reset_device(dev_handle.handle))
  File "/home/aida/micromamba/envs/odroid/lib/python3.11/site-packages/usb/backend/libusb1.py", line 604, in _check
    raise USBError(_strerror(ret), ret, _libusb_errno[ret])
usb.core.USBError: [Errno 2] Entity not found



In [53]:
# ========== CELL: SIMPLE STABILIZE (NO missing params) ==========
print("🔧 STABILIZE - Conservative gains only")

# 1. EMERGENCY STOP
odrv0.axis0.requested_state = 1  # IDLE
odrv0.axis1.requested_state = 1  # IDLE
time.sleep(1)

# 2. CONSERVATIVE GAINS (proven stable)
odrv0.axis0.controller.config.vel_gain = 0.05      # Quiet
odrv0.axis0.controller.config.vel_integrator_gain = 0.5
odrv0.axis0.controller.config.pos_gain = 20

odrv0.axis1.controller.config.vel_gain = 0.05
odrv0.axis1.controller.config.vel_integrator_gain = 0.5
odrv0.axis1.controller.config.pos_gain = 20

print("✅ CONSERVATIVE GAINS - NO NOISE!")
print(f"Axis0 vel_gain: {odrv0.axis0.controller.config.vel_gain}")


🔧 STABILIZE - Conservative gains only


AttributeError: 'RemoteObject' object has no attribute 'axis0'

In [54]:
# ========== SAFE RE-CALIBRATION ==========
print("🔄 RE-CALIBRATE (only existing params)")

# Reset calibration flags
try:
    odrv0.axis1.encoder.config.pre_calibrated = False
    odrv0.axis1.motor.config.pre_calibrated = False
except:
    pass

# ENCODER CALIBRATION
print("🔄 ENCODER CAL...")
odrv0.axis1.requested_state = 4  # ENCODER_OFFSET_CAL
time.sleep(25)
odrv0.axis1.encoder.config.pre_calibrated = True

print("✅ Axis1 encoder recalibrated!")


🔄 RE-CALIBRATE (only existing params)
🔄 ENCODER CAL...


AttributeError: 'RemoteObject' object has no attribute 'axis1'

In [55]:
# ========== QUIET TEST ==========
print("🔇 QUIET TEST")

odrv0.axis1.requested_state = 8  # CLOSED_LOOP
time.sleep(2)

# VERY SLOW test
print("Very slow ramp...")
for i in range(3):
    vel = i * 1.0  # 0→2 rev/s (super gentle)
    odrv0.axis1.controller.input_vel = vel
    print(f"Vel: {vel} rev/s")
    time.sleep(3)

odrv0.axis1.controller.input_vel = 0
print("🛑 Should be QUIET + NO oscillation!")


🔇 QUIET TEST


AttributeError: 'RemoteObject' object has no attribute 'axis1'